# PROCESSAMENTO DE LINGUAGEM NATURAL (PLN)
## Pré-processamento, Tokenização, Stopwords, Stemming e Lematização

**Objetivo:** 
1. Pré-processamento básico (lowercase, remoção de caracteres)
2. Tokenização
3. Remoção de stopwords
4. Stemming (RSLPStemmer)
5. Lematização
6. Salvar versões processadas

## 1. INSTALAÇÃO E IMPORTAÇÕES

In [1]:
# Instalação de bibliotecas NLP
!pip install nltk spacy pandas
!python -m spacy download pt_core_news_sm

     ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
     - -------------------------------------- 0.5/13.0 MB 5.6 MB/s eta 0:00:03
     --- ------------------------------------ 1.0/13.0 MB 2.6 MB/s eta 0:00:05
     --- ------------------------------------ 1.0/13.0 MB 2.6 MB/s eta 0:00:05
     --- ------------------------------------ 1.0/13.0 MB 2.6 MB/s eta 0:00:05
     --- ------------------------------------ 1.0/13.0 MB 2.6 MB/s eta 0:00:05
     --- ----------------------------------- 1.3/13.0 MB 882.6 kB/s eta 0:00:14
     ---- ---------------------------------- 1.6/13.0 MB 953.2 kB/s eta 0:00:12
     ---- ---------------------------------- 1.6/13.0 MB 953.2 kB/s eta 0:00:12
     ---- ---------------------------------- 1.6/13.0 MB 953.2 kB/s eta 0:00:12
     ----- --------------------------------- 1.8/13.0 MB 798.8 kB/s eta 0:00:14
     ------ -------------------------------- 2.1/13.0 MB 844.8 kB/s eta 0:00:13
     ------- ------------------------------- 2.4/13.0

In [2]:
import pandas as pd
import numpy as np
import re
import string
import os
from tqdm import tqdm
tqdm.pandas()

# NLTK
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

# Spacy para lematização
import spacy

# Downloads NLTK necessários
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('rslp', quiet=True)

print("✅ Bibliotecas importadas!")
print(f"✅ NLTK version: {nltk.__version__}")

✅ Bibliotecas importadas!
✅ NLTK version: 3.8.1


## 2. CARREGAR DADOS BALANCEADOS

In [3]:
# Carregar dados do Notebook 2
df = pd.read_csv('../data/processed/reviews_balanceadas.csv')

print("="*80)
print("📊 DADOS CARREGADOS")
print("="*80)
print(f"Total de reviews: {len(df):,}")
print(f"\nDistribuição de sentimentos:")
print(df['sentimento'].value_counts())
print(f"\nPrimeiras linhas:")
display(df.head())

📊 DADOS CARREGADOS
Total de reviews: 3,789

Distribuição de sentimentos:
sentimento
Negativo    1263
Neutro      1263
Positivo    1263
Name: count, dtype: int64

Primeiras linhas:


,texto,nota,data,app,sentimento,tamanho,palavras,ano_mes
0,Lixo,1,2019-10-08 21:42:36,Khan Academy,Negativo,4,1,2019-10
1,Muito caro,1,2024-04-12 18:45:11,Babbel,Negativo,10,2,2024-04
2,está fora de cervidor já usei antes e agora nã...,1,2026-04-05 16:34:47,Duolingo,Negativo,67,13,2026-04
3,"Esse app sempre foi muito bom, mas há algumas ...",1,2024-08-14 08:24:43,Google Classroom,Negativo,145,26,2024-08
4,"é útil? é. Mas e MUITO mal feito pqp, além de ...",2,2025-10-04 01:19:54,Google Classroom,Negativo,121,26,2025-10


## 3. PRÉ-PROCESSAMENTO

Etapas:
1. Lowercase
2. Remover URLs
3. Remover menções e hashtags
4. Remover números
5. Remover pontuação
6. Remover espaços extras

In [4]:
def preprocessar_basico(texto):
    """
    Pré-processamento básico do texto
    """
    # Converter para string
    texto = str(texto)
    
    # Lowercase
    texto = texto.lower()
    
    # Remover URLs
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)
    
    # Remover menções e hashtags
    texto = re.sub(r'@\w+|#\w+', '', texto)
    
    # Remover números
    texto = re.sub(r'\d+', '', texto)
    
    # Remover pontuação
    texto = texto.translate(str.maketrans('', '', string.punctuation))
    
    # Remover espaços extras
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

print("🔄 Aplicando pré-processamento básico...")
df['texto_preprocessado'] = df['texto'].progress_apply(preprocessar_basico)

print("\n✅ Pré-processamento básico concluído!")
print("\n📊 Exemplo:")
print(f"Original:       {df['texto'].iloc[0][:100]}...")
print(f"Preprocessado:  {df['texto_preprocessado'].iloc[0][:100]}...")

🔄 Aplicando pré-processamento básico...


100%|██████████| 3789/3789 [00:00<00:00, 14494.96it/s]


✅ Pré-processamento básico concluído!

📊 Exemplo:
Original:       Lixo...
Preprocessado:  lixo...


## 4. TOKENIZAÇÃO

Separar textos em tokens (palavras individuais)

In [5]:
print("🔄 Tokenizando textos...")
df['tokens'] = df['texto_preprocessado'].progress_apply(word_tokenize)

print("\n✅ Tokenização concluída!")
print("\n📊 Exemplo:")
print(f"Texto:  {df['texto_preprocessado'].iloc[0][:80]}...")
print(f"Tokens: {df['tokens'].iloc[0][:15]}")

🔄 Tokenizando textos...


100%|██████████| 3789/3789 [00:01<00:00, 3060.66it/s]


✅ Tokenização concluída!

📊 Exemplo:
Texto:  lixo...
Tokens: ['lixo']


## 5. REMOÇÃO DE STOPWORDS

Stopwords são palavras comuns sem significado relevante ("o", "a", "de", "para", etc.)

In [6]:
# Carregar stopwords do português
stop_words = set(stopwords.words('portuguese'))

print(f"📋 Total de stopwords em português: {len(stop_words)}")
print(f"\nExemplos: {list(stop_words)[:20]}")

# Função para remover stopwords
def remover_stopwords(tokens):
    return [palavra for palavra in tokens if palavra not in stop_words and len(palavra) > 2]

print("\n🔄 Removendo stopwords...")
df['tokens_sem_stopwords'] = df['tokens'].progress_apply(remover_stopwords)

print("\n✅ Stopwords removidas!")
print("\n📊 Comparação:")
print(f"Com stopwords:    {df['tokens'].iloc[0][:15]}")
print(f"Sem stopwords:    {df['tokens_sem_stopwords'].iloc[0][:15]}")

📋 Total de stopwords em português: 207

Exemplos: ['na', 'dos', 'lhes', 'eram', 'tiver', 'tínhamos', 'no', 'nossa', 'com', 'eles', 'haja', 'houveriam', 'seremos', 'eu', 'pelo', 'era', 'mas', 'aqueles', 'hajam', 'tiverem']

🔄 Removendo stopwords...


100%|██████████| 3789/3789 [00:00<00:00, 73166.23it/s]


✅ Stopwords removidas!

📊 Comparação:
Com stopwords:    ['lixo']
Sem stopwords:    ['lixo']


## 6. STEMMING (RSLPStemmer)

Reduz palavras ao seu radical/raiz.
Exemplo: "estudando", "estudar", "estudante" → "estud"

In [7]:
# Inicializar stemmer para português
stemmer = RSLPStemmer()

def aplicar_stemming(tokens):
    return [stemmer.stem(palavra) for palavra in tokens]

print("🔄 Aplicando stemming (RSLP)...")
df['tokens_stemmed'] = df['tokens_sem_stopwords'].progress_apply(aplicar_stemming)

# Juntar tokens em texto
df['texto_stemmed'] = df['tokens_stemmed'].apply(lambda x: ' '.join(x))

print("\n✅ Stemming concluído!")
print("\n📊 Exemplos de transformação:")
exemplos = [
    ['estudando', 'estudar', 'estudante'],
    ['aplicativo', 'aplicação', 'aplicar'],
    ['aprendendo', 'aprender', 'aprendizado']
]
for grupo in exemplos:
    stems = [stemmer.stem(p) for p in grupo]
    print(f"   {grupo} → {stems}")

print("\n📊 Exemplo de texto completo:")
print(f"Sem stemming: {' '.join(df['tokens_sem_stopwords'].iloc[0][:10])}")
print(f"Com stemming: {' '.join(df['tokens_stemmed'].iloc[0][:10])}")

🔄 Aplicando stemming (RSLP)...


100%|██████████| 3789/3789 [00:03<00:00, 1060.60it/s]


✅ Stemming concluído!

📊 Exemplos de transformação:
   ['estudando', 'estudar', 'estudante'] → ['estud', 'estud', 'estud']
   ['aplicativo', 'aplicação', 'aplicar'] → ['aplic', 'aplic', 'aplic']
   ['aprendendo', 'aprender', 'aprendizado'] → ['aprend', 'aprend', 'aprend']

📊 Exemplo de texto completo:
Sem stemming: lixo
Com stemming: lix


## 7. LEMATIZAÇÃO (Spacy)

Similar ao stemming, mas retorna palavras válidas (lemas).
Exemplo: "estudando", "estudar" → "estudar" (verbo no infinitivo)

In [8]:
# Carregar modelo Spacy para português
print("🔄 Carregando modelo Spacy (pt_core_news_sm)...")
nlp = spacy.load('pt_core_news_sm')

def lematizar_texto(texto):
    """
    Aplica lematização usando Spacy
    """
    # Processar com Spacy
    doc = nlp(texto)
    
    # Extrair lemas, removendo stopwords e pontuação
    lemas = [token.lemma_ for token in doc 
             if not token.is_stop and not token.is_punct and len(token.text) > 2]
    
    return lemas

print("\n🔄 Aplicando lematização (pode demorar alguns minutos)...")
df['tokens_lemmatized'] = df['texto_preprocessado'].progress_apply(lematizar_texto)

# Juntar tokens em texto
df['texto_lemmatized'] = df['tokens_lemmatized'].apply(lambda x: ' '.join(x))

print("\n✅ Lematização concluída!")
print("\n📊 Comparação Stemming vs Lematização:")
print(f"Stemming:     {' '.join(df['tokens_stemmed'].iloc[0][:10])}")
print(f"Lematização:  {' '.join(df['tokens_lemmatized'].iloc[0][:10])}")

🔄 Carregando modelo Spacy (pt_core_news_sm)...

🔄 Aplicando lematização (pode demorar alguns minutos)...


100%|██████████| 3789/3789 [01:04<00:00, 58.87it/s]


✅ Lematização concluída!

📊 Comparação Stemming vs Lematização:
Stemming:     lix
Lematização:  lixo


## 8. ESTATÍSTICAS DO PROCESSAMENTO

In [9]:
# Calcular estatísticas
stats = pd.DataFrame({
    'Etapa': [
        'Original',
        'Pré-processado',
        'Tokens',
        'Sem Stopwords',
        'Stemming',
        'Lematização'
    ],
    'Tamanho Médio': [
        df['texto'].str.len().mean(),
        df['texto_preprocessado'].str.len().mean(),
        df['tokens'].apply(len).mean(),
        df['tokens_sem_stopwords'].apply(len).mean(),
        df['tokens_stemmed'].apply(len).mean(),
        df['tokens_lemmatized'].apply(len).mean()
    ]
})

print("="*80)
print("📊 ESTATÍSTICAS DO PROCESSAMENTO")
print("="*80)
display(stats)

# Redução de vocabulário
vocab_original = set(' '.join(df['texto_preprocessado']).split())
vocab_stemmed = set(' '.join(df['texto_stemmed']).split())
vocab_lemmatized = set(' '.join(df['texto_lemmatized']).split())

print(f"\n📚 Tamanho do vocabulário:")
print(f"   Original:     {len(vocab_original):,} palavras únicas")
print(f"   Stemming:     {len(vocab_stemmed):,} palavras únicas ({(1-len(vocab_stemmed)/len(vocab_original))*100:.1f}% redução)")
print(f"   Lematização:  {len(vocab_lemmatized):,} palavras únicas ({(1-len(vocab_lemmatized)/len(vocab_original))*100:.1f}% redução)")

📊 ESTATÍSTICAS DO PROCESSAMENTO


,Etapa,Tamanho Médio
0,Original,100.125363
1,Pré-processado,97.443917
2,Tokens,17.466086
3,Sem Stopwords,9.545263
4,Stemming,9.545263
5,Lematização,7.764318



📚 Tamanho do vocabulário:
   Original:     6,580 palavras únicas
   Stemming:     3,614 palavras únicas (45.1% redução)
   Lematização:  4,583 palavras únicas (30.3% redução)


## 9. SALVAR DADOS PROCESSADOS

In [10]:
# Selecionar colunas importantes para salvar
df_save = df[[
    'texto',
    'nota',
    'data',
    'app',
    'sentimento',
    'texto_preprocessado',
    'texto_stemmed',
    'texto_lemmatized'
]].copy()

# Salvar
output_file = '../data/processed/reviews_pln.csv'
df_save.to_csv(output_file, index=False, encoding='utf-8-sig')

print("="*80)
print("💾 DADOS COM PLN SALVOS")
print("="*80)
print(f"📁 Arquivo: {output_file}")
print(f"📊 Linhas: {len(df_save):,}")
print(f"📊 Colunas: {list(df_save.columns)}")
print(f"💽 Tamanho: {os.path.getsize(output_file) / (1024*1024):.2f} MB")

print(f"\n✅ PLN finalizado! Prossiga para o Notebook 4 (TF-IDF + ML)")

💾 DADOS COM PLN SALVOS
📁 Arquivo: ../data/processed/reviews_pln.csv
📊 Linhas: 3,789
📊 Colunas: ['texto', 'nota', 'data', 'app', 'sentimento', 'texto_preprocessado', 'texto_stemmed', 'texto_lemmatized']
💽 Tamanho: 1.32 MB

✅ PLN finalizado! Prossiga para o Notebook 4 (TF-IDF + ML)


## 10. RESUMO DO PROCESSAMENTO

In [11]:
print("="*80)
print("📋 RESUMO DO NOTEBOOK 3 - PLN")
print("="*80)

print(f"\n✅ Etapas concluídas:")
print(f"   1. ✅ Pré-processamento básico")
print(f"   2. ✅ Tokenização")
print(f"   3. ✅ Remoção de stopwords")
print(f"   4. ✅ Stemming (RSLPStemmer)")
print(f"   5. ✅ Lematização (Spacy)")
print(f"   6. ✅ Dados salvos")

print(f"\n📊 Resultado:")
print(f"   Vocabulário reduzido: {len(vocab_original):,} → {len(vocab_stemmed):,} palavras")
print(f"   Duas versões processadas: Stemming e Lematização")

print(f"\n📁 Arquivo gerado:")
print(f"   • ../data/processed/reviews_pln.csv")

print(f"\n➡️  Próximos notebooks:")
print(f"   • Notebook 4: TF-IDF + 5 Modelos ML (usa stemming)")
print(f"   • Notebook 5: Métodos Lexicais (usa lematização)")

📋 RESUMO DO NOTEBOOK 3 - PLN

✅ Etapas concluídas:
   1. ✅ Pré-processamento básico
   2. ✅ Tokenização
   3. ✅ Remoção de stopwords
   4. ✅ Stemming (RSLPStemmer)
   5. ✅ Lematização (Spacy)
   6. ✅ Dados salvos

📊 Resultado:
   Vocabulário reduzido: 6,580 → 3,614 palavras
   Duas versões processadas: Stemming e Lematização

📁 Arquivo gerado:
   • ../data/processed/reviews_pln.csv

➡️  Próximos notebooks:
   • Notebook 4: TF-IDF + 5 Modelos ML (usa stemming)
   • Notebook 5: Métodos Lexicais (usa lematização)
